# 04.08 - DeepLabV3 ResNet50 versus ResNet101 evaluation

**Notebook type:** Solution notebook with completed exercises, smoke checks, and test cases.

**Daily output:** DeepLabV3 backbone benchmark and ensemble evidence.

Benchmark the official ResNet50 and ResNet101 DeepLabV3 architectures on identical bounded inputs, then compare prepared validation logits and their ensemble with per-class IoU.

## Core Ideas

Backbone depth changes parameter count, compute, memory, and feature capacity. Fair comparisons use identical preprocessing, input batches, output schemas, devices, and warm-up policy. Runtime from one tiny CPU run demonstrates mechanics, not a universal ranking. Logit ensembling is valid only when class order and spatial alignment match.

In [ ]:
import time
import numpy as np
import pandas as pd
import torch
from torchvision.models.segmentation import deeplabv3_resnet50, deeplabv3_resnet101

SEED = 4
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Prepared Benchmark Batch and Validation Logits

Two 32×32 images benchmark architecture inference. Separate deterministic three-class logits and masks provide interpretable metric and ensemble evidence independent of random offline weights.

In [ ]:
benchmark_images = torch.randn(2, 3, 32, 32)
validation_masks = torch.zeros((4, 16, 16), dtype=torch.int64)
validation_masks[0:2, 3:11, 2:8] = 1
validation_masks[2:4, 5:13, 8:14] = 2
base_logits = torch.nn.functional.one_hot(validation_masks, num_classes=3).permute(0, 3, 1, 2).float() * 3.0
resnet50_logits = base_logits + 0.7 * torch.randn_like(base_logits)
resnet101_logits = base_logits + 0.55 * torch.randn_like(base_logits)
resnet50_logits[:, 1, 0:2, 0:3] += 3.5
resnet101_logits[:, 2, 14:16, 13:16] += 3.2
print("benchmark/logits:", benchmark_images.shape, resnet50_logits.shape)

## Exercise 04-A: Build both official architectures

Disable both segmentation and backbone weights so validation never downloads a checkpoint.

**Return structure — `build_deeplab_backbones`:** A dictionary with keys `resnet50` and `resnet101`; each value is a Torchvision `DeepLabV3` with `num_classes` output channels on `device`.

In [ ]:
def build_deeplab_backbones(num_classes=3, device=DEVICE):
    common = {"weights": None, "weights_backbone": None, "num_classes": int(num_classes), "aux_loss": False}
    return {"resnet50": deeplabv3_resnet50(**common).to(device), "resnet101": deeplabv3_resnet101(**common).to(device)}


# Smoke check: construct both exact architectures offline.
backbone_models = build_deeplab_backbones()
print({name: type(model.backbone).__name__ for name, model in backbone_models.items()})

## Exercise 04-B: Benchmark one shared batch

Run each model in evaluation/inference mode and normalize outputs to one evidence table.

**Return structure — `benchmark_deeplab_models`:** A tuple `(outputs, table)`. `outputs` maps model name to CPU float32 logits `[N,C,H,W]`; `table` has columns `model`, `parameters`, `output_shape`, and `runtime_seconds`.

In [ ]:
def benchmark_deeplab_models(models, images, device=DEVICE):
    outputs, rows = {}, []
    for name, model in models.items():
        model.eval(); start = time.perf_counter()
        with torch.inference_mode(): logits = model(images.to(device))["out"]
        runtime = time.perf_counter() - start; outputs[name] = logits.cpu()
        rows.append({"model": name, "parameters": sum(parameter.numel() for parameter in model.parameters()), "output_shape": tuple(logits.shape), "runtime_seconds": runtime})
    return outputs, pd.DataFrame(rows)


# Smoke check: benchmark identical bounded inputs.
architecture_outputs, architecture_table = benchmark_deeplab_models(backbone_models, benchmark_images)
print(architecture_table.to_string(index=False))

## Exercise 04-C: Calculate per-class IoU

Calculate one IoU per class and retain classes absent from both masks as perfect by explicit convention.

**Return structure — `multiclass_iou`:** A dictionary with `per_class_iou` (`list[float]` length `num_classes`), Python float `mean_iou`, and integer `pixel_count`.

In [ ]:
def multiclass_iou(predictions, targets, num_classes=3):
    predictions, targets = predictions.cpu().long(), targets.cpu().long(); values = []
    for class_id in range(int(num_classes)):
        predicted_class, target_class = predictions == class_id, targets == class_id
        intersection = int((predicted_class & target_class).sum()); union = int((predicted_class | target_class).sum())
        values.append(float(intersection / union) if union else 1.0)
    return {"per_class_iou": values, "mean_iou": float(np.mean(values)), "pixel_count": int(targets.numel())}


# Smoke check: score each prepared model's logits.
resnet50_scores = multiclass_iou(resnet50_logits.argmax(dim=1), validation_masks)
resnet101_scores = multiclass_iou(resnet101_logits.argmax(dim=1), validation_masks)
print("R50/R101 mIoU:", resnet50_scores["mean_iou"], resnet101_scores["mean_iou"])

## Exercise 04-D: Ensemble aligned logits

Average logits only after verifying identical shapes, then expose each model and ensemble in one table.

**Return structure — `segmentation_ensemble_evidence`:** A tuple `(ensemble_predictions, table)`. Predictions are CPU int64 `[N,H,W]`; table has `strategy`, `pixel_count`, `per_class_iou`, `mean_iou`, and `delta_from_resnet50`.

In [ ]:
def segmentation_ensemble_evidence(logits_by_model, targets):
    shapes = {tuple(logits.shape) for logits in logits_by_model.values()}
    if len(shapes) != 1:
        raise ValueError("all logits must share class order and shape")
    rows = []
    for name, logits in logits_by_model.items():
        score = multiclass_iou(logits.argmax(dim=1), targets)
        rows.append({"strategy": name, **score})
    ensemble_logits = torch.stack(list(logits_by_model.values())).mean(dim=0)
    ensemble_predictions = ensemble_logits.argmax(dim=1).cpu().to(torch.int64)
    rows.append({"strategy": "logit_ensemble", **multiclass_iou(ensemble_predictions, targets)})
    table = pd.DataFrame(rows); table["delta_from_resnet50"] = table["mean_iou"] - float(table.iloc[0]["mean_iou"])
    return ensemble_predictions, table


# Smoke check and complete prepared-validation evidence.
ensemble_predictions, ensemble_table = segmentation_ensemble_evidence({"resnet50": resnet50_logits, "resnet101": resnet101_logits}, validation_masks)
print(ensemble_table.to_string(index=False))

## Test Cases

**Return structure — `run_day04_tests`:** Returns `None`; assertions and `Day 04 tests passed` communicate success.

In [ ]:
def run_day04_tests():
    assert set(backbone_models) == {"resnet50", "resnet101"}
    assert architecture_table.shape == (2, 4)
    assert set(architecture_outputs) == {"resnet50", "resnet101"}
    assert all(output.shape == (2, 3, 32, 32) for output in architecture_outputs.values())
    parameter_map = dict(zip(architecture_table["model"], architecture_table["parameters"]))
    assert parameter_map["resnet101"] > parameter_map["resnet50"]
    assert len(resnet50_scores["per_class_iou"]) == len(resnet101_scores["per_class_iou"]) == 3
    assert ensemble_predictions.shape == validation_masks.shape
    assert ensemble_table.shape == (3, 5) and float(ensemble_table.iloc[0]["delta_from_resnet50"]) == 0.0
    print("Day 04 tests passed")


run_day04_tests()

## Day 04 Checklist

- [ ] Build exact ResNet50 and ResNet101 DeepLabV3 architectures.
- [ ] Use identical inputs and output schemas.
- [ ] Compare parameters and latency without asserting a universal winner.
- [ ] Ensemble only aligned class logits.
- [ ] Run the test cases.